# DELTA DIY MRI Workshop — Environment Setup

This notebook sets up a clean environment for the workshop notebooks.

It follows a simple path:

1. Detect operating system and CPU.
2. Check or install Python 3.11.x.
3. Create a clean virtual environment.
4. Install workshop packages.
5. Select the notebook kernel.
6. Run smoke tests for PyPulseq and MRZeroCore.

**Recommended macOS choice:** use the Python installer from `python.org`, not Homebrew Python, for the workshop environment. This avoids the Homebrew `pyexpat/libexpat` issue observed on macOS 26.x during testing.


In [ ]:
# Cell 1 — Detect operating system and CPU architecture

import platform
import sys
from pathlib import Path

print("Cell 1 — System detection")
print("-------------------------")

os_name = platform.system()
machine = platform.machine().lower()

if os_name == "Darwin":
    if machine in ["arm64", "aarch64"]:
        setup_category = "macOS Apple Silicon"
    elif machine in ["x86_64", "amd64"]:
        setup_category = "macOS Intel"
    else:
        setup_category = "macOS unknown CPU"
elif os_name == "Windows":
    setup_category = "Windows"
elif os_name == "Linux":
    setup_category = "Linux"
else:
    setup_category = "Unknown"

print("Current notebook Python:")
print("  Executable:", sys.executable)
print("  Version   :", sys.version.split()[0])

print("\nSystem:")
print("  OS        :", os_name)
print("  Platform  :", platform.platform())
print("  Machine   :", platform.machine())
print("  Processor :", platform.processor())
print("  Category  :", setup_category)

# The workshop venv will be created in the current folder.
repo_dir = Path.cwd()
venv_name = ".diy-mri-workshop"

print("\nWorkshop folder:") # Make sure this is ./adelpha 
print("  Current folder:", repo_dir)
print("  Venv folder   :", repo_dir / venv_name)


Cell 1 — System detection
-------------------------
Current notebook Python:
  Executable: /opt/homebrew/opt/python@3.10/bin/python3.10
  Version   : 3.10.20

System:
  OS        : Darwin
  Platform  : macOS-26.2-arm64-arm-64bit
  Machine   : arm64
  Processor : arm
  Category  : macOS Apple Silicon

Workshop folder:
  Current folder: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks
  Venv folder   : /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks/.diy-mri-workshop


## Cell 2 — Python 3.11 installation instructions

Run Cell 2 to print the correct Python 3.11 check and install instructions for your system.

For macOS, the workshop-recommended path is the `python.org` installer. During beta testing, this avoided Homebrew-specific Python issues.


In [2]:
# Cell 2 — Check for Python 3.11 and print install instructions

import platform
import subprocess
import shutil
from pathlib import Path

print("Cell 2 — Python 3.11 check")
print("--------------------------")

os_name = platform.system()

if os_name == "Darwin":
    python311_candidate = "/usr/local/bin/python3.11"
    install_instructions = """
macOS install instructions
--------------------------

Recommended for this workshop:
1. Open: https://www.python.org/downloads/
2. Download a Python 3.11.x macOS universal2 installer.
3. Run the .pkg installer.
4. Open a new Terminal window.
5. Verify:

   /usr/local/bin/python3.11 --version
   /usr/local/bin/python3.11 -c "import xml.parsers.expat as expat; print('pyexpat OK')"
   /usr/local/bin/python3.11 -c "import platform; print(platform.mac_ver())"

Do not use Homebrew Python for the workshop if these checks fail.
"""
    check_cmd = [python311_candidate, "--version"]

elif os_name == "Windows":
    python311_candidate = "py -3.11"
    install_instructions = r"""
Windows install instructions
----------------------------

Recommended for this workshop:
1. Open: https://www.python.org/downloads/
2. Download a Python 3.11.x Windows installer.
3. Run the installer.
4. Select: Add python.exe to PATH
5. Open a new Command Prompt.
6. Verify:

   py -3.11 --version
"""
    check_cmd = ["py", "-3.11", "--version"]

else:
    python311_candidate = "python3.11"
    install_instructions = """
Linux / fallback install instructions
-------------------------------------

Install Python 3.11 using your system package manager, then verify:

   python3.11 --version
"""
    check_cmd = ["python3.11", "--version"]

print("Detected OS:", os_name)
print("Recommended Python 3.11 command:", python311_candidate)

print("\nChecking Python 3.11 availability...")
try:
    result = subprocess.run(check_cmd, capture_output=True, text=True, timeout=10)
    version_text = result.stdout.strip() or result.stderr.strip()
    print("Return code:", result.returncode)
    print("Output     :", version_text)

    if result.returncode == 0 and "3.11" in version_text:
        print("\nPASS: Python 3.11 is available.")
        python311_ready = True
    else:
        print("\nPython 3.11 was not detected.")
        python311_ready = False
        print(install_instructions)

except Exception as exc:
    print("\nPython 3.11 was not detected.")
    python311_ready = False
    print("Reason:", repr(exc))
    print(install_instructions)

print("\npython311_ready:", python311_ready)


Cell 2 — Python 3.11 check
--------------------------
Detected OS: Darwin
Recommended Python 3.11 command: /usr/local/bin/python3.11

Checking Python 3.11 availability...
Return code: 0
Output     : Python 3.11.9

PASS: Python 3.11 is available.

python311_ready: True


## Cell 3 — Create the workshop virtual environment

This cell prints terminal commands. Run the printed commands in Terminal or Command Prompt.

After the environment is created, return to this notebook and continue to Cell 4.


In [3]:
# Cell 3 — Print commands to create a clean workshop virtual environment

import platform
from pathlib import Path

print("Cell 3 — Create virtual environment")
print("-----------------------------------")

os_name = platform.system()
repo_dir = Path.cwd()
venv_name = ".diy-mri-workshop"

print("Detected OS:", os_name)
print("Workshop folder:", repo_dir)
print("Venv folder:", repo_dir / venv_name)

print("\nRun these commands in Terminal / Command Prompt.")
print("-----------------------------------------------")

if os_name == "Darwin":
    print(f"""
cd "{repo_dir}"

deactivate 2>/dev/null || true

# Clear any old pip compatibility settings from previous debugging attempts.
unset PIP_USE_DEPRECATED

rm -rf {venv_name}

/usr/local/bin/python3.11 -m venv {venv_name}

{venv_name}/bin/python --version
{venv_name}/bin/python -m pip --version
{venv_name}/bin/python -c "import xml.parsers.expat as expat; print('pyexpat OK')"
{venv_name}/bin/python -c "import platform; print('mac_ver:', platform.mac_ver())"
""")

elif os_name == "Windows":
    print(fr"""
cd /d "{repo_dir}"

rmdir /s /q {venv_name}

py -3.11 -m venv {venv_name}

{venv_name}\Scripts\python --version
{venv_name}\Scripts\python -m pip --version
""")

else:
    print(f"""
cd "{repo_dir}"

deactivate 2>/dev/null || true

rm -rf {venv_name}

python3.11 -m venv {venv_name}

{venv_name}/bin/python --version
{venv_name}/bin/python -m pip --version
""")

print("Expected:")
print("  Python 3.11.x")
print("  pip ... from .../.diy-mri-workshop/... (python 3.11)")
print("  On macOS: pyexpat OK")


Cell 3 — Create virtual environment
-----------------------------------
Detected OS: Darwin
Workshop folder: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks
Venv folder: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks/.diy-mri-workshop

Run these commands in Terminal / Command Prompt.
-----------------------------------------------

cd "/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks"

deactivate 2>/dev/null || true

# Clear any old pip compatibility settings from previous debugging attempts.
unset PIP_USE_DEPRECATED

rm -rf .diy-mri-workshop

/usr/local/bin/python3.11 -m venv .diy-mri-workshop

.diy-mri-workshop/bin/python --version
.diy-mri-workshop/bin/python -m pip --version
.diy-mri-workshop/bin/python -c "import xml.parsers.expat as expat; print('pyexpat OK')"
.diy-mri-workshop/bin/python -c "import platform; print('mac_ver:', platform.mac_ver())"

Expected:


## Cell 4 — Install workshop packages

This cell prints terminal commands. It does not require the environment to be activated. It installs packages directly into the virtual environment.


In [5]:
# Cell 4 — Print commands to install workshop packages

import platform
from pathlib import Path

print("Cell 4 — Install workshop packages")
print("----------------------------------")

os_name = platform.system()
repo_dir = Path.cwd()
venv_name = ".diy-mri-workshop"

packages = [
    "numpy<2",
    "matplotlib",
    "scipy",
    "torch",
    "pypulseq==1.4.2.post1",
    "MRzeroCore",
    "jupyter",
    "ipykernel",
]

print("Packages:")
for p in packages:
    print(" ", p)

print("\nRun these commands in Terminal / Command Prompt.")
print("-----------------------------------------------")

if os_name == "Darwin":
    py = f"{venv_name}/bin/python"
    print(f"""
cd "{repo_dir}"

unset PIP_USE_DEPRECATED

{py} -m pip install --upgrade pip setuptools wheel

{py} -m pip install \\
  "numpy<2" \\
  "matplotlib" \\
  "scipy" \\
  "torch" \\
  "pypulseq==1.4.2.post1" \\
  "MRzeroCore" \\
  "jupyter" \\
  "ipykernel"

{py} -m ipykernel install --user \\
  --name diy-mri-workshop \\
  --display-name "DIY MRI Workshop Python 3.11"

{py} -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
{py} -c "import numpy as np; print('NumPy OK:', np.__version__)"
{py} -c "import matplotlib; print('Matplotlib OK:', matplotlib.__version__)"
{py} -c "import scipy; print('SciPy OK:', scipy.__version__)"
{py} -c "import torch; print('Torch OK:', torch.__version__)"
{py} -c "import pypulseq as pp; print('PyPulseq OK:', pp.__file__)"
{py} -c "import MRzeroCore as mr0; print('MRZeroCore OK:', mr0.__file__)"
{py} -c "import platform; print('mac_ver:', platform.mac_ver())"
""")

elif os_name == "Windows":
    py = fr"{venv_name}\Scripts\python"
    print(fr"""
cd /d "{repo_dir}"

{py} -m pip install --upgrade pip setuptools wheel

{py} -m pip install ^
  "numpy<2" ^
  "matplotlib" ^
  "scipy" ^
  "torch" ^
  "pypulseq==1.4.2.post1" ^
  "MRzeroCore" ^
  "jupyter" ^
  "ipykernel"

{py} -m ipykernel install --user ^
  --name diy-mri-workshop ^
  --display-name "DIY MRI Workshop Python 3.11"

{py} -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
{py} -c "import numpy as np; print('NumPy OK:', np.__version__)"
{py} -c "import matplotlib; print('Matplotlib OK:', matplotlib.__version__)"
{py} -c "import scipy; print('SciPy OK:', scipy.__version__)"
{py} -c "import torch; print('Torch OK:', torch.__version__)"
{py} -c "import pypulseq as pp; print('PyPulseq OK:', pp.__file__)"
{py} -c "import MRzeroCore as mr0; print('MRZeroCore OK:', mr0.__file__)"
""")

else:
    py = f"{venv_name}/bin/python"
    print(f"""
cd "{repo_dir}"

{py} -m pip install --upgrade pip setuptools wheel

{py} -m pip install \\
  "numpy<2" \\
  "matplotlib" \\
  "scipy" \\
  "torch" \\
  "pypulseq==1.4.2.post1" \\
  "MRzeroCore" \\
  "jupyter" \\
  "ipykernel"

{py} -m ipykernel install --user \\
  --name diy-mri-workshop \\
  --display-name "DIY MRI Workshop Python 3.11"

{py} -c "import numpy as np; print('NumPy OK:', np.__version__)"
{py} -c "import pypulseq as pp; print('PyPulseq OK:', pp.__file__)"
{py} -c "import MRzeroCore as mr0; print('MRZeroCore OK:', mr0.__file__)"
""")

print("After this succeeds, select this kernel:")
print("  DIY MRI Workshop Python 3.11")


Cell 4 — Install workshop packages
----------------------------------
Packages:
  numpy<2
  matplotlib
  scipy
  torch
  pypulseq==1.4.2.post1
  MRzeroCore
  jupyter
  ipykernel

Run these commands in Terminal / Command Prompt.
-----------------------------------------------

cd "/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks"

unset PIP_USE_DEPRECATED

.diy-mri-workshop/bin/python -m pip install --upgrade pip setuptools wheel

.diy-mri-workshop/bin/python -m pip install \
  "numpy<2" \
  "matplotlib" \
  "scipy" \
  "torch" \
  "pypulseq==1.4.2.post1" \
  "MRzeroCore" \
  "jupyter" \
  "ipykernel"

.diy-mri-workshop/bin/python -m ipykernel install --user \
  --name diy-mri-workshop \
  --display-name "DIY MRI Workshop Python 3.11"

.diy-mri-workshop/bin/python -c "import sys; print('Python executable:', sys.executable); print('Python version:', sys.version)"
.diy-mri-workshop/bin/python -c "import numpy as np; print('NumPy OK:', np.__version__

## Cell 5 — Verify the selected notebook kernel

After installing packages, select the kernel named:

`DIY MRI Workshop Python 3.11`

Then restart the notebook kernel and run Cell 5.


In [1]:
# Cell 5 — Verify selected notebook kernel

import sys
import platform

print("Cell 5 — Notebook kernel verification")
print("-------------------------------------")

print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("Platform         :", platform.platform())
print("mac_ver          :", platform.mac_ver())

expected_env_name = ".diy-mri-workshop"

if expected_env_name in sys.executable:
    print("\nPASS: This notebook is using the workshop virtual environment.")
else:
    print("\nWARNING: This notebook may not be using the workshop virtual environment.")
    print("Select the kernel:")
    print("  DIY MRI Workshop Python 3.11")
    print("Then restart the notebook kernel and rerun this cell.")


Cell 5 — Notebook kernel verification
-------------------------------------
Python executable: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/bin/python
Python version   : 3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
Platform         : macOS-26.2-arm64-arm-64bit
mac_ver          : ('26.2', ('', '', ''), 'arm64')

PASS: This notebook is using the workshop virtual environment.


In [2]:
# Cell 6 — Package import smoke test

print("Cell 6 — Package import smoke test")
print("----------------------------------")

import sys
import platform

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])
print("mac_ver          :", platform.mac_ver())

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
import torch
import pypulseq as pp
import MRzeroCore as mr0

print("\nCore packages")
print("-------------")
print("NumPy      :", np.__version__)
print("Matplotlib :", matplotlib.__version__)
print("SciPy      :", scipy.__version__)
print("Torch      :", torch.__version__)

print("\nMRI packages")
print("------------")
print("PyPulseq   :", pp.__file__)
print("MRZeroCore :", mr0.__file__)

print("\nPASS: All workshop packages imported successfully.")


Cell 6 — Package import smoke test
----------------------------------
Python executable: /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/bin/python
Python version   : 3.11.9
mac_ver          : ('26.2', ('', '', ''), 'arm64')

Core packages
-------------
NumPy      : 1.26.4
Matplotlib : 3.11.1
SciPy      : 1.17.1
Torch      : 2.14.0

MRI packages
------------
PyPulseq   : /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/lib/python3.11/site-packages/pypulseq/__init__.py
MRZeroCore : /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/lib/python3.11/site-packages/MRzeroCore/__init__.py

PASS: All workshop packages imported successfully.


In [4]:
# Cell 7 — PyPulseq sequence export smoke test
#
# This checks that PyPulseq can create and export a simple .seq file.
# This is not a scanner-ready MRI sequence.

print("Cell 7 — PyPulseq sequence export smoke test")
print("--------------------------------------------")

import numpy as np
import pypulseq as pp
from pathlib import Path

BLOCK_RASTER_TIME = 10e-6

def round_to_raster(t, raster=BLOCK_RASTER_TIME):
    """Round time in seconds to nearest raster point."""
    t_rounded = round(t / raster) * raster
    if t_rounded < 0:
        raise ValueError(f"Negative delay after rounding: {t_rounded*1e6:.1f} µs")
    return t_rounded

system = pp.Opts(
    max_grad=10,
    grad_unit="mT/m",
    max_slew=50,
    slew_unit="T/m/s",
    rf_ringdown_time=20e-6,
    rf_dead_time=100e-6,
    adc_dead_time=10e-6,
)

seq = pp.Sequence(system)

rf_duration = round_to_raster(200e-6)
delay_duration = round_to_raster(10e-3)

rf = pp.make_block_pulse(
    flip_angle=np.pi / 2,
    duration=rf_duration,
    delay=system.rf_dead_time,
    system=system,
)

seq.add_block(rf)
seq.add_block(pp.make_delay(delay_duration))

ok, error_report = seq.check_timing()

if ok:
    print("PASS: PyPulseq timing check passed.")
else:
    print("WARNING: PyPulseq timing check failed.")
    for err in error_report:
        print(err)

out_file = Path("pypulseq_smoke_test.seq")

if ok:
    seq.write(str(out_file))
    print(f"PASS: Wrote {out_file.resolve()}")
else:
    print("Skipping seq.write() because timing check failed.")

if out_file.exists():
    print("PASS: .seq file exists.")
    print("File size:", out_file.stat().st_size, "bytes")
else:
    print("WARNING: .seq file was not created.")


Cell 7 — PyPulseq sequence export smoke test
--------------------------------------------
PASS: PyPulseq timing check passed.
PASS: Wrote /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/console/notebooks/pypulseq_smoke_test.seq
PASS: .seq file exists.
File size: 1048 bytes


In [5]:
# Cell 8 — MRZeroCore smoke test and minimal API check

print("Cell 8 — MRZeroCore smoke test")
print("------------------------------")

import inspect
from importlib.metadata import version
import MRzeroCore as mr0

print("MRZeroCore imported successfully.")
print("MRZeroCore version:", version("MRzeroCore"))
print("MRZeroCore path   :", mr0.__file__)

expected_names = [
    "Sequence",
    "VoxelGridPhantom",
    "CustomVoxelPhantom",
    "compute_graph",
    "execute_graph",
    "reco_adjoint",
]

print("\nExpected object check")
print("---------------------")

all_found = True

for name in expected_names:
    if hasattr(mr0, name):
        obj = getattr(mr0, name)
        print(f"PASS: mr0.{name} found")
        try:
            print("      Signature:", inspect.signature(obj))
        except Exception:
            pass
    else:
        all_found = False
        print(f"WARNING: mr0.{name} not found")

if all_found:
    print("\nPASS: MRZeroCore has the expected objects for the workshop notebooks.")
else:
    print("\nWARNING: MRZeroCore imported, but the API differs from the tested setup.")


Cell 8 — MRZeroCore smoke test
------------------------------
MRZeroCore imported successfully.
MRZeroCore version: 1.0.8
MRZeroCore path   : /Users/sairamgeethanath/Documents/Contributions/Tools/Projects/adelpha/.diy-mri-workshop/lib/python3.11/site-packages/MRzeroCore/__init__.py

Expected object check
---------------------
PASS: mr0.Sequence found
      Signature: (repetitions: 'Iterable[Repetition]' = [], normalized_grads: 'bool' = True)
PASS: mr0.VoxelGridPhantom found
      Signature: (PD: 'torch.Tensor', T1: 'torch.Tensor', T2: 'torch.Tensor', T2dash: 'torch.Tensor', D: 'torch.Tensor', B0: 'torch.Tensor', B1: 'torch.Tensor', coil_sens: 'torch.Tensor', affine: 'torch.Tensor', phantom_motion=None, voxel_motion=None, tissue_masks: 'Optional[Dict[str, torch.Tensor]]' = None) -> 'None'
PASS: mr0.CustomVoxelPhantom found
      Signature: (pos: 'list[list[float]] | torch.Tensor', PD: 'float | list[float] | torch.Tensor' = 1.0, T1: 'float | list[float] | torch.Tensor' = 1.5, T2: 'float 

## Setup complete

The environment is ready when these are true:

- Cell 5 says the notebook is using `.diy-mri-workshop`.
- Cell 6 imports all packages.
- Cell 7 writes `pypulseq_smoke_test.seq`.
- Cell 8 finds the expected MRZeroCore objects.

You can now run the RF spin echo, 1D SE projection, and 2D SE notebooks.
